# 13 - Unified Customer Profile
Merge segmentation, churn, and inactivity data into a 360-degree customer view.

In [1]:
import pandas as pd, numpy as np
import plotly.express as px
import warnings; warnings.filterwarnings('ignore')
SEED=42
PRIMARY='#635BFF'; RISK='#E74C3C'; SAFE='#27AE60'; NEUTRAL='#3498DB'; WARNING='#F39C12'; TEMPLATE='plotly_white'


In [2]:
# Load sources
segments = pd.read_csv('data/processed/customer_with_segments.csv')
print(f"Segments shape: {segments.shape}")
print(f"Columns: {list(segments.columns)}")

churn = pd.read_csv('data/processed/churn_predictions.csv')
print(f"\nChurn shape: {churn.shape}")

inactivity = pd.read_csv('data/processed/inactivity_scores.csv')
print(f"\nInactivity shape: {inactivity.shape}")


Segments shape: (10127, 26)
Columns: ['CLIENTNUM', 'Attrition_Flag', 'Customer_Age', 'Gender', 'Dependent_count', 'Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category', 'Months_on_book', 'Total_Relationship_Count', 'Months_Inactive_12_mon', 'Contacts_Count_12_mon', 'Credit_Limit', 'Total_Revolving_Bal', 'Avg_Open_To_Buy', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt', 'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio', 'churn_label', 'cluster_id', 'segment_name', 'pca_1', 'pca_2']

Churn shape: (10127, 20)

Inactivity shape: (10127, 4)


In [3]:
# Merge
profile = segments.copy()
profile = profile.merge(churn[['CLIENTNUM','churn_probability']], on='CLIENTNUM', how='left')
profile = profile.merge(inactivity[['CLIENTNUM','activity_score','activity_category','future_churn_candidate']],
                        on='CLIENTNUM', how='left')
print(f"Merged profile shape: {profile.shape}")

# Derive churn_risk_label
def risk_label(p):
    """Assign risk label from churn probability."""
    if p >= 0.70: return 'High Risk'
    elif p >= 0.40: return 'Medium Risk'
    else: return 'Loyal'

profile['churn_risk_label'] = profile['churn_probability'].apply(risk_label)


Merged profile shape: (10127, 30)


In [4]:
# Select final columns
final_cols = ['CLIENTNUM','Customer_Age','Gender','Income_Category','Card_Category',
              'Education_Level','Marital_Status','Credit_Limit','Total_Trans_Amt',
              'Total_Trans_Ct','Avg_Utilization_Ratio','Months_Inactive_12_mon',
              'segment_name','churn_probability','churn_risk_label',
              'activity_score','activity_category','future_churn_candidate']
available = [c for c in final_cols if c in profile.columns]
profile = profile[available]
print(f"Final profile columns ({len(available)}): {available}")


Final profile columns (18): ['CLIENTNUM', 'Customer_Age', 'Gender', 'Income_Category', 'Card_Category', 'Education_Level', 'Marital_Status', 'Credit_Limit', 'Total_Trans_Amt', 'Total_Trans_Ct', 'Avg_Utilization_Ratio', 'Months_Inactive_12_mon', 'segment_name', 'churn_probability', 'churn_risk_label', 'activity_score', 'activity_category', 'future_churn_candidate']


In [5]:
# Summary statistics
print("=== Segment Distribution ===")
print(profile['segment_name'].value_counts())
print("\n=== Mean Churn Probability by Segment ===")
print(profile.groupby('segment_name')['churn_probability'].mean().round(4))
print("\n=== Mean Activity Score by Segment ===")
print(profile.groupby('segment_name')['activity_score'].mean().round(4))
print(f"\nFuture churn candidates: {profile['future_churn_candidate'].sum()}")


=== Segment Distribution ===
segment_name
Deal Hunters         3051
At-Risk Customers    2829
Silent Users         1981
Premium Customers    1446
Daily Spenders        820
Name: count, dtype: int64

=== Mean Churn Probability by Segment ===
segment_name
At-Risk Customers    0.3696
Daily Spenders       0.0092
Deal Hunters         0.0216
Premium Customers    0.1671
Silent Users         0.2391
Name: churn_probability, dtype: float64

=== Mean Activity Score by Segment ===
segment_name
At-Risk Customers    0.3356
Daily Spenders       0.6446
Deal Hunters         0.4991
Premium Customers    0.3960
Silent Users         0.3820
Name: activity_score, dtype: float64

Future churn candidates: 2197


In [6]:
seg = profile['segment_name'].value_counts().reset_index(); seg.columns=['segment','count']
fig = px.pie(seg, values='count', names='segment', hole=0.4, template=TEMPLATE,
             title='Customer Segment Distribution')
fig.show()


In [7]:
churn_seg = profile.groupby('segment_name')['churn_probability'].mean().reset_index()
fig = px.bar(churn_seg, x='segment_name', y='churn_probability', color_discrete_sequence=[RISK],
             template=TEMPLATE, title='Mean Churn Probability by Segment')
fig.show()


In [8]:
crl = profile['churn_risk_label'].value_counts().reset_index(); crl.columns=['label','count']
fig = px.pie(crl, values='count', names='label', hole=0.3,
             color='label', color_discrete_map={'High Risk':RISK,'Medium Risk':WARNING,'Loyal':SAFE},
             template=TEMPLATE, title='Churn Risk Label Distribution')
fig.show()


In [9]:
ac = profile['activity_category'].value_counts().reset_index(); ac.columns=['category','count']
fig = px.bar(ac, x='category', y='count', color='category',
             color_discrete_map={'Active':SAFE,'Moderately Active':NEUTRAL,'Inactive':WARNING,'High Risk':RISK},
             template=TEMPLATE, title='Activity Category Distribution')
fig.show()


In [10]:
fc = profile['future_churn_candidate'].value_counts().reset_index(); fc.columns=['candidate','count']
fc['candidate'] = fc['candidate'].map({True:'Yes',False:'No'})
fig = px.bar(fc, x='candidate', y='count', color='candidate',
             color_discrete_map={'Yes':RISK,'No':SAFE},
             template=TEMPLATE, title='Future Churn Candidates')
fig.show()


In [11]:
# Null audit
print("Null counts per column:")
print(profile.isnull().sum())


Null counts per column:
CLIENTNUM                 0
Customer_Age              0
Gender                    0
Income_Category           0
Card_Category             0
Education_Level           0
Marital_Status            0
Credit_Limit              0
Total_Trans_Amt           0
Total_Trans_Ct            0
Avg_Utilization_Ratio     0
Months_Inactive_12_mon    0
segment_name              0
churn_probability         0
churn_risk_label          0
activity_score            0
activity_category         0
future_churn_candidate    0
dtype: int64


In [12]:
# Save
profile.to_csv('data/processed/unified_customer_profile.csv', index=False)
print(f"Saved unified_customer_profile.csv - shape: {profile.shape}")

# Overwrite customer_features.parquet
profile.to_parquet('data/features/customer_features.parquet', index=False)
print("Saved customer_features.parquet (overwritten)")


Saved unified_customer_profile.csv - shape: (10127, 18)
Saved customer_features.parquet (overwritten)
